# Token Embedding and Distributed Representation

> The previous two sections completed the Tokenizer: text is split into tokens, and each token is assigned an integer ID. But an ID is just an ID -- there is no magnitude relationship between 7 and 3, and the model cannot tell from IDs alone whether two words are similar.
>
> This section introduces Embedding: mapping discrete token IDs into a continuous vector space. We start from "why vectors are needed," build intuition about word vectors step by step, and finally assemble the Embedding layer actually used in training.

The idea behind Embedding is to map each token ID to a fixed-length vector of real numbers. Different dimensions within the same vector can express different features -- some dimensions might capture "whether it's an animal," some might capture "size," and some might capture "abstractness." These features are not hand-picked; they are discovered by the model during training. This way, cat and dog have similar values across most dimensions, while algorithm differs from them in every dimension -- the distance between two vectors directly reflects how semantically close they are.


## 1. From IDs to Vectors

To make this intuition more concrete, let's look at a hand-scored table. Suppose we score each word on four dimensions:

| | Size | Fluffiness | Friendliness | Independence |
|:---|:---|:---|:---|:---|
| cat | 2 | 8 | 6 | 9 |
| dog | 5 | 9 | 9 | 3 |
| algorithm | 0 | 0 | 0 | 0 |

Each row is a set of numbers, which can be written as a vector:

```text
cat       -> [2, 8, 6, 9]
dog       -> [5, 9, 9, 3]
algorithm -> [0, 0, 0, 0]
```

The more dimensions, the finer the description. Cat and dog are relatively close on each dimension, while algorithm is completely different from both. What Embedding does is essentially the same thing -- except these features are not hand-picked, but learned by the model during training. Using multiple continuous values to jointly describe a token, where the values themselves are learned from data -- this is the core idea of Embedding.


### From Color RGB to Word Vectors

Above we used four dimensions to describe cat, dog, and algorithm, which works much better than a single ID. This is in fact the core idea of Embedding: using multiple numbers to describe something, rather than just giving it an ID.

But there's a question: why should cat's "fluffiness" be scored 8? What's the standard for measuring "friendliness"? In actual Embeddings, these values are not set by humans -- they are learned by the model. To understand how the model learns, we can start with an everyday example.

Colors can be described in two ways. One is to give each color a name: cobalt blue, carmine red, coral orange... The more names, the more words needed, and just looking at names, you can't tell how close two colors are. The other way is to describe colors using three RGB values, like (201, 23, 30). Three numbers suffice, and judging similarity is intuitive -- the closer two colors' values, the more similar they look. (201, 23, 30) and (180, 20, 40) are both red-ish, while (23, 180, 201) is blue-ish -- just compute the numerical distance.

Word representation faces the same problem. Giving each word an ID (ID=5 is cat, ID=12 is dog) is like giving colors names -- there's no magnitude relationship between IDs, and you can't express that "cat and dog are close, but cat and algorithm are far apart." But if each word has a vector like RGB, the similarity between words can be directly measured by numerical distance.

This way of describing words with multiple numbers is called distributed representation. Unlike one-hot -- which uses an extremely long vector where 10,000 words require 10,000 dimensions, with only 1 position being 1 and the rest all 0 -- distributed representation uses only a few hundred dimensions, each being a real number, and the distance between vectors directly reflects semantic closeness.

But where do the values in the vectors come from? They can't be filled in randomly; they need to be learned from data. To understand the learning process, we first need to understand one thing: how context determines a word's meaning.


### Context Determines Meaning

There is a lot of research on representing words as vectors. Looking carefully at this research, you'll find that almost all important methods are based on one simple idea: the meaning of a word is formed by the words surrounding it.

This idea is straightforward. Words themselves don't have inherent meaning; a word's meaning is formed by the context it appears in. Words with similar meanings frequently appear in similar contexts. For example:

```text
I drink beer.    We drink wine.
I guzzle beer.   We guzzle wine.
```

Drink often appears near beverages, and guzzle also often appears near beverages -- drink and guzzle have similar contexts. Based on this observation, we can infer that guzzle and drink are synonyms (guzzle means "to drink greedily").

Here, context refers to the words surrounding a word of interest. The size of the context (i.e., the number of surrounding words) is called the window size. A window size of 1 means the context includes 1 word on each side; a window size of 2 means the context includes 2 words on each side, and so on.

```text
Corpus: You say goodbye and I say hello.

Window size = 2, focus word = goodbye:
  You say goodbye and I say hello.
      <---- context ---->

  goodbye's context words = {You, say, and, I} (2 on the left + 2 on the right)
```

This section only handles the case where the number of words on the left and right are equal, without considering sentence delimiters.


In [ ]:
# Co-occurrence matrix demo: build a word-context matrix by hand with a mini corpus
# Using the corpus introduced above: "You say goodbye and I say hello."
corpus = "You say goodbye and I say hello ."
# For simplicity, split by spaces and lowercase everything
tokens = corpus.lower().split()
vocab = sorted(set(tokens))
V = len(vocab)

# token -> index mapping
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print(f"Corpus: {corpus}")
print(f"Tokenized: {tokens}")
print(f"Vocabulary ({V} words): {vocab}\n")

# Window size = 2 (2 words on each side)
window_size = 2

# Initialize VxV zero matrix: rows = focus word, columns = context word
co_matrix = [[0] * V for _ in range(V)]

# Iterate over each position as the focus word
for center_pos in range(len(tokens)):
    center_word = tokens[center_pos]
    center_idx = word2idx[center_word]
    # Take window_size words on each side as context
    start = max(0, center_pos - window_size)
    end = min(len(tokens), center_pos + window_size + 1)
    for ctx_pos in range(start, end):
        if ctx_pos == center_pos:
            continue
        ctx_word = tokens[ctx_pos]
        ctx_idx = word2idx[ctx_word]
        co_matrix[center_idx][ctx_idx] += 1

# Print co-occurrence matrix
header = "       " + "  ".join(f"{w:>6s}" for w in vocab)
print("Co-occurrence matrix (rows = focus word, columns = context word):")
print(header)
for i, row in enumerate(co_matrix):
    row_str = "  ".join(f"{val:6d}" for val in row)
    print(f"{idx2word[i]:>6s} | {row_str}")

print(f"\nKey observation:")
print(f"  Each word is described by a {V}-dimensional row vector -- this row is its distributed representation.")
print(f"  Words with similar meanings (e.g., say and goodbye are both high-frequency)")
print(f"  tend to have similar context distribution patterns.")
print(f"  But the problem is -- {V} words require a {V}x{V} = {V*V} count matrix,")
print(f"  and each dimension is just raw frequency, unable to capture indirect semantic associations.")


The code above shows a 7x7 co-occurrence matrix. Each row is a word's vector -- the row for "say" is [1, 2, 1, 1, 1, 0, 1], indicating how many times it encountered and, goodbye, hello, and other words in context. This way of describing words using context counts is the most primitive form of distributed representation.

The co-occurrence matrix intuitively demonstrates the idea that "similar context leads to similar vectors." But it has two obvious limitations: the matrix size is the square of the vocabulary size (a 100k-word vocabulary means a 100k x 100k matrix), and each dimension is just a raw count, unable to capture more complex semantic relationships.

Modern LLMs no longer use this count-based representation. Instead, they learn low-dimensional dense vectors through training the Embedding layer. Next, let's look at the specific implementation of the Embedding layer.


## 2. Embedding Lookup

The dimension of Embedding (usually written as `embed_dim` or `d_model`) determines how long a vector each token is represented as. Different model families use different parameter names in their config.json. The table below summarizes the common naming conventions:

| Meaning | GPT-2 style | Modern models (LLaMA / Qwen / DeepSeek / Gemma / Phi) | Common in papers / teaching |
|:---|:---|:---|:---|
| Embedding dimension | `n_embd` | `hidden_size` | `d_model` |
| Number of attention heads | `n_head` | `num_attention_heads` | `n_heads` |
| Number of Transformer layers | `n_layer` | `num_hidden_layers` | `n_layers` |
| Vocabulary size | `vocab_size` | `vocab_size` | `V` |
| FFN intermediate dimension | 4 x n_embd | `intermediate_size` | `d_ff` |
| Maximum sequence length | `n_positions` | `max_position_embeddings` | `max_seq_len` |
| Number of KV heads (GQA) | -- | `num_key_value_heads` | `n_kv_heads` |

GPT-2 is an early open-source model; its `n_embd` / `n_layer` / `n_head` naming is common in GPT-2 and reproduction projects like nanoGPT. The original 2017 Transformer paper uses the `d_model` / `d_ff` notation. In public configs of HuggingFace Transformers, many modern decoder-only models (LLaMA, Qwen, Mistral, DeepSeek, Gemma, Phi) use field names like `hidden_size` / `num_hidden_layers` / `num_attention_heads`; some model families keep their own naming or nest it inside `text_config`. When you see `d_model` in a paper, you usually look for `hidden_size` in the HuggingFace config -- here they all refer to the model dimension that flows through the residual stream.

One point that's easy to find confusing: why is the Embedding dimension called `hidden_size` ("hidden layer size") in HuggingFace configs, rather than `embedding_size`? The reason is the Transformer's residual stream design -- from the Embedding layer to the final output layer, the main vector dimension passed between layers stays consistent:

```text
token ID -> Embedding -> [hidden_size] -> Self-Attention -> [hidden_size] -> FFN -> [hidden_size] -> ... -> lm_head
```

The Embedding outputs a `hidden_size`-dimensional vector, which passes through Self-Attention and FFN layer by layer with its dimension unchanged, until the final lm_head projects it back to the vocabulary size. Because the dimension is the same from start to end, `hidden_size` simultaneously describes the Embedding's output dimension and each layer's hidden vector dimension -- they are the same number. The original Transformer paper calls it `d_model` (model dimension), with the same meaning of "the dimension that runs through the entire model." The FFN's intermediate layer temporarily widens to `intermediate_size` (usually about 4x), but that's just a temporary expansion inside the FFN; the output returns to `hidden_size`.

The table below summarizes the Embedding configurations of mainstream models. Model names link to the corresponding HuggingFace config.json for verification:

| Model | Vocab Size | Embedding Dim | Embedding Parameters |
|:---|:---|:---|:---|
| [GPT-2 small](https://huggingface.co/openai-community/gpt2/blob/main/config.json) | 50,257 | 768 | ~39M |
| [Qwen2.5 0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B/blob/main/config.json) | 151,936 | 896 | ~136M |
| [Gemma 3 1B](https://huggingface.co/unsloth/gemma-3-1b-pt/blob/main/config.json) | 262,144 | 1,152 | ~302M |
| [GPT-2 medium](https://huggingface.co/openai-community/gpt2-medium/blob/main/config.json) | 50,257 | 1,024 | ~51M |
| [Gemma 3 4B](https://huggingface.co/unsloth/gemma-3-4b-it/blob/main/config.json) | 262,208 | 2,560 | ~671M |
| [LLaMA 2 7B](https://huggingface.co/NousResearch/Llama-2-7b-hf/blob/b3ced1bf8cac12ec77f23961af1bda9f439b8113/config.json) | 32,000 | 4,096 | ~131M |
| [Mistral 7B](https://huggingface.co/mistralai/Mistral-7B-v0.1/blob/29b3489dc8f0210f34d2a4bda08ccc072cd20d09/config.json) | 32,000 | 4,096 | ~131M |
| [Qwen2.5 7B](https://huggingface.co/Qwen/Qwen2.5-7B/blob/main/config.json) | 152,064 | 3,584 | ~545M |
| [LLaMA 3 8B](https://huggingface.co/NousResearch/Meta-Llama-3-8B/blob/main/config.json) | 128,256 | 4,096 | ~525M |
| [DeepSeek V3](https://huggingface.co/deepseek-ai/DeepSeek-V3/blob/main/config.json) | 129,280 | 7,168 | ~927M |
| [LLaMA 3 70B](https://huggingface.co/NousResearch/Meta-Llama-3-70B/blob/main/config.json) | 128,256 | 8,192 | ~1,051M |
| [Qwen2.5 72B](https://huggingface.co/Qwen/Qwen2.5-72B/blob/main/config.json) | 152,064 | 8,192 | ~1,246M |

Note: Gemma 3 4B is a multimodal model. When opening its config, look for `text_config.vocab_size` and `text_config.hidden_size`. For other models, `vocab_size` / `hidden_size` is usually at the top level; GPT-2's Embedding dimension field is called `n_embd`.

Several patterns emerge from the table:

- **Vocabulary size varies significantly.** GPT-2's 50k-token vocabulary was sufficient at the time, but modern multilingual models (Qwen, Gemma) need 150k-260k tokens to cover more languages. LLaMA and Mistral are more conservative, staying at 32k-128k.
- **Dimensions grow with model scale.** Small models use a few hundred dimensions, 7B-level models use 3K-4K dimensions, and 70B+ models use 7K-8K dimensions. Higher dimensions allow vectors to hold more information.
- **Embedding parameter count = vocab_size x d_model.** Note that while Qwen2.5 0.5B has only 500M total parameters, its Embedding layer alone has 136M parameters -- over a quarter of the total. Small models with large vocabularies often have Embedding as the biggest parameter component.

`nn.Embedding` is essentially a matrix like this:

```text
Matrix shape: [vocab_size, d_model]

Row 0 -> vector for token 0
Row 1 -> vector for token 1
Row 2 -> vector for token 2
...
```

Give the Embedding layer a token ID, and it retrieves the corresponding row. These vectors start out random and are continuously adjusted by the model during training. After training, words that frequently appear in similar contexts will have vectors that are closer together.


In [ ]:
import torch
import torch.nn as nn

# Simulate a mini vocabulary; Embedding = vocab_size x embed_dim matrix
vocab = ["the", "cat", "sat", "on", "mat", "dog", "log"]
vocab_size = len(vocab)
embed_dim = 4

embedding = nn.Embedding(vocab_size, embed_dim)

print(f"Vocabulary size: {vocab_size}, Embedding dimension: {embed_dim}")
print(f"Embedding weight shape: {embedding.weight.shape}  <- a {vocab_size}x{embed_dim} matrix")
print(f"\nFirst 3 rows (random initialization):\n{embedding.weight[:3]}")


In [ ]:
import torch

# Lookup: given a set of token IDs, retrieve the corresponding vectors
sentence_ids = torch.tensor([0, 1, 2, 3, 0, 4])  # "the cat sat on the mat"
vectors = embedding(sentence_ids)                  # lookup -> [6, 4]

print(f"token IDs: {sentence_ids.tolist()}  ->  {[vocab[i] for i in sentence_ids.tolist()]}")
print(f"Output shape: {vectors.shape}  <- [{len(sentence_ids)} tokens, each {embed_dim} dimensions]")
print()

# Examine each position
for i, (tid, vec) in enumerate(zip(sentence_ids.tolist(), vectors)):
    print(f"  Position {i}: '{vocab[tid]}' (ID={tid}) -> {vec.tolist()}")

# Key observation: positions 0 and 4 are both 'the', vectors are identical
# -> The same token gets the same vector regardless of where it appears
print(f"\nKey observation: positions 0 and 4 are both token 'the', and the retrieved vectors are identical")
print(f"-> The same token gets the same vector regardless of where it appears")
print(f"-> The model still cannot distinguish word order -- this is the problem Positional Encoding will solve in the next section")


**How Embeddings Are Trained**

Vectors don't magically possess semantics -- they need to be learned from data. There are two main approaches in practice.

The first is pre-trained word vectors, represented by Word2Vec and GloVe. The idea is to train Embedding separately, then use it as fixed input for downstream models.

Take Word2Vec's Skip-gram as an example: take a word (like "cat"), use its vector to predict the surrounding words within a window ("sat", "on", "mat"). When predictions are wrong, the vector gets adjusted. After training, words that frequently appear in similar contexts (cat, dog) end up with vectors closer together. GloVe has a similar idea but doesn't train through prediction -- instead, it directly leverages global word co-occurrence statistics.

The advantage of this approach is that the Embedding is trained once and can be reused. The disadvantage is that the Embedding is fixed during downstream tasks and cannot be fine-tuned for the task. It's like laying a foundation first, then building on top -- if the foundation's position doesn't match the building's structure, there's no way to go back and adjust.

The second approach is end-to-end training, which is what modern LLMs use. The Embedding matrix is no longer trained separately but serves as part of the model's parameters, updated together with the other Transformer layers through backpropagation. Specifically: during training, the model receives a batch of token IDs, looks up vectors through the Embedding layer, and the vectors pass through several Transformer layers before the output layer computes the loss. The loss computes gradients for all model parameters, and backpropagation propagates all the way back to the Embedding layer -- each row's vector values are adjusted by the gradient. This process is identical to any Linear layer in the middle of the Transformer: no freezing, no special handling, no separate learning rate.

This can be verified from two angles. From a code perspective, nanoGPT (Karpathy's GPT-2 reproduction) has a `configure_optimizers` method that passes all `requires_grad=True` parameters to AdamW. Embedding weights, as a 2D matrix, participate in weight decay as usual -- the comments explicitly note "embeddings decay." From an academic research perspective, an EMNLP 2024 paper specifically analyzes gradient behavior in the Embedding layer during pre-training, concluding that "the token embedding layer's gradient has the largest norm" -- under Pre-LN architecture, shallow layer gradients are larger than deep layers, and Embedding as the first layer receives the strongest gradients. A COLM 2025 paper proposes adding LayerNorm after Embedding to prevent gradient explosion. If Embedding didn't participate in training, these studies wouldn't exist.

Early in training, vectors are randomly distributed; mid-training, synonym vectors start to cluster; late in training, the vector space forms structures consistent with language statistics. This is not pre-designed -- it's the natural result of gradient updates.

Note a common source of confusion: the Embedding layer discussed here is internal to the LLM -- a [vocab_size, d_model] lookup matrix that takes token IDs as input and outputs token vectors. In scenarios like retrieval-augmented generation (RAG), you'll also encounter standalone Embedding models like BGE, E5, Jina Embeddings, and Qwen3 Embedding -- these are complete models that encode entire text passages into sentence vectors, serving a different role.

Below is a runnable code example demonstrating the end-to-end training process. We use a mini Embedding matrix and one Linear layer to simulate a simplified Transformer, run a few training steps, and observe how the matrix changes.


In [ ]:
# End-to-end training demo: how the Embedding matrix updates through backpropagation
import torch
import torch.nn as nn

torch.manual_seed(42)

# Use a mini vocabulary for the demo
vocab_size, d_model = 10, 4
embedding = nn.Embedding(vocab_size, d_model)

# Use one Linear layer to simulate a simplified "Transformer"
linear = nn.Linear(d_model, vocab_size, bias=False)

print("Embedding matrix before training:")
print(embedding.weight.data)
print(f"\nShape: {embedding.weight.shape}  <- [vocab_size={vocab_size}, d_model={d_model}]")

# Create fake data: 4 samples, 3 tokens each
input_ids = torch.randint(0, vocab_size, (4, 3))
targets = torch.randint(0, vocab_size, (4, 3))

optimizer = torch.optim.SGD(
    list(embedding.parameters()) + list(linear.parameters()), lr=0.1
)

print(f"\ninput_ids shape: {input_ids.shape}  <- [batch=4, seq_len=3]")
print(f"targets shape:   {targets.shape}")
print(f"\nTraining...\n")

for step in range(5):
    vectors = embedding(input_ids)           # [4, 3, d_model] -- lookup vectors
    logits = linear(vectors)                  # [4, 3, vocab_size] -- project to vocab space
    loss = nn.functional.cross_entropy(
        logits.view(-1, vocab_size),          # [12, vocab_size]
        targets.view(-1)                      # [12]
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"  step {step+1}: loss = {loss.item():.4f}")

print(f"\nEmbedding matrix after training:")
print(embedding.weight.data)

# Key observation: compare changes in the same row before and after training
print(f"\nKey observation:")
print(f"  Every row of the Embedding matrix has changed through gradient updates.")
print(f"  This is end-to-end training -- Embedding as a model parameter,")
print(f"  updated together with other Transformer layers, driven by the loss.")
print(f"  We'll see the complete training flow when implementing Mini-GPT later.")


## 3. Embedding Training Practices in Industry

The previous `nn.Embedding` lookup explained the concept clearly. But in real large language model training, there are several engineering decisions that directly affect parameter efficiency and training stability. Let's look at each one, referencing practices from actual open-source code.

**Weight Tying**

A model uses vocabulary-sized matrices in two places. At the beginning is the Embedding layer: it takes token IDs as input and looks up vectors. At the end is the output layer (lm_head): it takes vectors as input and outputs vocabulary-sized logits. Both matrices have the same shape, [vocab_size, d_model].

GPT-2's approach is to have these two matrices share the same weights -- lm_head.weight directly points to wte.weight. This saves parameters equal to the vocabulary size: with vocab_size=50257 and d_model=768, it saves about 39 million parameters.

This technique hasn't disappeared, but not all large models use it. In public open-source configurations, LLaMA series defaults to `tie_word_embeddings=False`, meaning the input Embedding and output lm_head are independent; while some small or parameter-sensitive models still use tying to save parameters.

So a more accurate statement is: **weight tying is an optional design**. Small models often use it to save parameters; many large models separate the two matrices, letting the input representation and output classifier learn independently. GPT-3's full weights and implementation are not public, so it shouldn't be cited as a definitive example. References: [HuggingFace LLaMA config](https://huggingface.co/docs/transformers/v4.50.0/en/model_doc/llama), [LLaMA 3.2 1B config example](https://meta-pytorch.org/torchtune/0.6/generated/torchtune.models.llama3_2.llama3_2_1b.html).

**Weight Decay Scope**

Weight decay is a constraint the optimizer applies during each parameter update: first shrink the parameter proportionally (w <- w - lr x lambda x w), then subtract the current gradient direction. This is equivalent to adding a lambda*||w||^2/2 penalty to the loss, applying a stronger pull-back force on large weights to prevent parameter values from diverging.

A common intuition is: each row of the Embedding matrix is a token's semantic vector. If we pull the row vector toward the origin (i.e., decay each component's absolute value), the semantic vector's "length" is artificially shortened -- and semantics shouldn't have an artificial strong/weak ranking. Following this logic, Embedding weights should be excluded from weight decay.

But this intuition lacks experimental support. Actual open-source code does the opposite.

nanoGPT (Karpathy) has only one rule for parameter grouping: parameters with `p.dim() >= 2` participate in decay. Embedding weights have shape [vocab_size, d_model], which is a 2D matrix with dim=2, so they fall into the decay group.

HuggingFace Trainer and many training scripts commonly exclude bias and LayerNorm/RMSNorm weights from decay; but parameter grouping is fundamentally determined by the training script, not enforced by the framework. In other words, whether Embedding participates in weight decay depends on the specific recipe. For teaching purposes, one reliable conclusion suffices: **bias/Norm usually don't decay; whether Embedding decays depends on the parameter grouping code.** References: [HuggingFace Trainer docs](https://huggingface.co/docs/transformers/main_classes/trainer), [HuggingFace forum discussion](https://discuss.huggingface.co/t/parameter-groups-and-gpt2-layernorm/4239).

**Initialization Standard Deviation**

`nn.Embedding` weights are a [vocab_size, d_model] matrix, initialized by default with N(0, 1) -- each element is independently drawn from a normal distribution with mean 0 and variance 1 (i.e., standard deviation 1). Each row is a token's vector with length equal to d_model.

Taking d_model=768 as an example. Take one row v = (x_1, x_2, ..., x_768), where each x_i ~ N(0, 1). The vector's magnitude (L2 norm) is defined as ||v|| = sqrt(x_1^2 + x_2^2 + ... + x_768^2). The calculation goes like this: for a single component, for N(0, 1), E[x_i^2] = Var(x_i) = 1 (when the mean is 0, the expectation of the square equals the variance). With 768 independent components accumulated, the expected sum of squares is about 768, so the magnitude falls around sqrt(768) ~= 28.

What does 28 mean? Although most components are between -1 and 1, across 768 dimensions, the distance from the vector's endpoint to the origin approaches 28. Taking Self-Attention as an example, two vectors with magnitude 28 doing a dot product produces a result with variance around 768, which after softmax leads to saturated gradients that hinder stable convergence early in training.

GPT-2's approach is simple: "A simple weight initialization of N(0, 0.02) was sufficient." -- all linear layer and Embedding weights are initialized with standard deviation 0.02, regardless of d_model. LLaMA follows the same approach, with initializer_range typically around 0.02.

Using the same method to calculate the magnitude. Each component x_i ~ N(0, 0.02), E[x_i^2] = 0.02^2 = 0.0004. With d_model independent components accumulated, the expected magnitude is sqrt(d_model x 0.0004) = 0.02 x sqrt(d_model):

| Initialization | d_model=768 | d_model=4096 |
|:---|:---|:---|
| N(0, 1) (PyTorch default) | magnitude ~= 28 | magnitude ~= 64 |
| N(0, 0.02) (GPT-2) | magnitude ~= 0.55 | magnitude ~= 1.28 |

0.02 compresses the magnitude to around 1, and when d_model goes from 768 to 4096, the magnitude only goes from 0.55 to 1.28 -- a small change. Combined with LayerNorm in each Transformer layer, which renormalizes activations to a standard range, the requirement for initialization precision is further reduced. This is why a fixed 0.02 works across different model scales.

**Embedding Under Mixed Precision**

Training with FP16/BF16 can accelerate computation and save memory. PyTorch's AMP (Automatic Mixed Precision) has built-in handling for this:

- `autocast` decides which operations use low precision and which stay in high precision based on device, dtype, and operator policy; don't simply remember it as "Embedding is always FP32."
- Many mixed precision training setups maintain higher-precision parameters/states in the optimizer or training framework, but the specific implementation depends on AMP, the optimizer, and the distributed framework.

If using BF16 (supported on A100/H100), the dynamic range is the same as FP32, making it generally less prone to overflow than FP16. Reference: [PyTorch AMP docs](https://docs.pytorch.org/docs/stable/amp.html).

The training demo below references Karpathy's [nanoGPT](https://github.com/karpathy/nanoGPT) project. Specifically: the parameter grouping logic (`p.dim() >= 2` participates in weight decay) and N(0, 0.02) initialization both come from [model.py](https://github.com/karpathy/nanoGPT/blob/master/model.py)'s `configure_optimizers` and `_init_weights` methods. We load real text from HuggingFace, build a 3000-word mini vocabulary, train an Embedding model, and finally use t-SNE to visualize the distribution of word vectors before and after training.


In [ ]:
# Load real text from HuggingFace, build vocabulary and training sequences
# Following nanoGPT data processing: raw text -> tokenize -> count word frequencies -> vocabulary -> token ID sequences

import torch
import torch.nn as nn
import re
from collections import Counter

# First time running, install with: !pip install datasets -q
from datasets import load_dataset

torch.manual_seed(42)

# Step 1: Load wikitext-2 training set (Wikipedia English corpus)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
print(f"Dataset samples: {len(dataset)}")

# Step 2: Simple tokenization -- extract lowercase English words only
def simple_tokenize(text):
    return re.findall(r'\b[a-z]+\b', text.lower())

# Step 3: Count word frequencies
word_counter = Counter()
for example in dataset:
    if example["text"] and len(example["text"].strip()) > 0:
        word_counter.update(simple_tokenize(example["text"]))

print(f"Total words in corpus: {sum(word_counter.values()):,}")
print(f"Unique words:   {len(word_counter):,}")

# Step 4: Take the most common words to build a mini vocabulary
vocab_size = 3000
most_common = word_counter.most_common(vocab_size)
vocab_words = [w for w, _ in most_common]
word2idx = {w: i for i, w in enumerate(vocab_words)}
idx2word = {i: w for w, i in word2idx.items()}

print(f"Vocabulary size: {vocab_size}")
print(f"First 20 words: {vocab_words[:20]}")
print(f"Words 200-220:  {vocab_words[200:220]}")

# Step 5: Convert entire corpus to token ID sequences (keep only words in vocabulary)
all_tokens = []
for example in dataset:
    if example["text"]:
        words = simple_tokenize(example["text"])
        ids = [word2idx[w] for w in words if w in word2idx]
        all_tokens.extend(ids)

print(f"\nValid tokens: {len(all_tokens):,}")

# Step 6: Build training sequences -- GPT's training objective: given N tokens, predict the next one
seq_len = 12
sequences = []
for i in range(0, len(all_tokens) - seq_len - 1, seq_len // 2):
    sequences.append(all_tokens[i:i + seq_len + 1])

# Take a subset of sequences for the demo
n_sequences = 3000
data_tensor = torch.tensor(sequences[:n_sequences])
x_data = data_tensor[:, :seq_len]       # Input: first seq_len tokens
y_data = data_tensor[:, 1:seq_len + 1]  # Target: predict next token at each position

print(f"Training samples: {n_sequences}, each of length: {seq_len}")
print(f"x shape: {list(x_data.shape)}, y shape: {list(y_data.shape)}")
print(f"\nSample examples (decoded as words):")
for i in range(2):
    in_words = [idx2word[tid.item()] for tid in x_data[i]]
    tgt_words = [idx2word[tid.item()] for tid in y_data[i]]
    print(f"  [{i}] {' '.join(in_words):60s}")
    print(f"      -> predict: {' '.join(tgt_words)}")


In [ ]:
# Build mini GPT-style model + nanoGPT parameter grouping + pre-training t-SNE visualization

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Configure matplotlib to render labels correctly
plt.rcParams['axes.unicode_minus'] = False

embed_dim = 32

# Step 1: Define model -- Embedding + lm_head, the minimal GPT skeleton
class MiniLLM(nn.Module):
    """Mini GPT-style model: Embedding -> lm_head, demonstrating Embedding training"""
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)
        # GPT-2/LLaMA style: all weights initialized with N(0, 0.02)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.lm_head.weight, mean=0.0, std=0.02)

    def forward(self, x):
        # x: [batch, seq_len] -> vecs: [batch, seq_len, embed_dim]
        # -> logits: [batch, seq_len, vocab_size]
        return self.lm_head(self.embedding(x))

model = MiniLLM(vocab_size, embed_dim)

total = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total:,} (Embedding: {vocab_size * embed_dim:,}, "
      f"lm_head: {vocab_size * embed_dim:,})")
print(f"Initial std: {model.embedding.weight.std().item():.4f}  <- N(0, 0.02)")

# Step 2: nanoGPT-style parameter grouping -- 2D matrices participate in weight_decay
decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if param.dim() >= 2:
        decay_params.append(param)       # Embedding, lm_head are 2D matrices
    else:
        no_decay_params.append(param)    # bias, LayerNorm (none in this example)

n_decay = sum(p.numel() for p in decay_params)
n_no_decay = sum(p.numel() for p in no_decay_params)
print(f"\nParticipating in weight_decay:   {n_decay:,}  <- Embedding is here")
print(f"Not participating in weight_decay: {n_no_decay:,}")

optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': 0.1},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=0.01)

# Step 3: Select visualization words -- four groups with clear semantics, observe position changes
viz_groups_def = {
    "numbers": ["one", "two", "three", "four", "five"],
    "time": ["year", "time", "day", "age", "period"],
    "places": ["city", "town", "area", "county", "state"],
    "people": ["son", "father", "mother", "wife", "family"],
}
color_map = {"numbers": "#E74C3C", "time": "#3498DB",
             "places": "#2ECC71", "people": "#F39C12"}

# Keep only words that actually exist in the vocabulary
viz_words, viz_indices, viz_colors, viz_groups_actual = [], [], [], {}
for group_name, words in viz_groups_def.items():
    idx_list = []
    for w in words:
        if w in word2idx:
            viz_words.append(w)
            viz_indices.append(word2idx[w])
            viz_colors.append(color_map[group_name])
            idx_list.append(word2idx[w])
    if idx_list:
        viz_groups_actual[group_name] = idx_list

print(f"\nVisualization words ({len(viz_words)} total):")
for name, idx_list in viz_groups_actual.items():
    words = [idx2word[i] for i in idx_list]
    print(f"  {name}: {', '.join(words)}")

# Step 4: Pre-training t-SNE -- record initial Embedding for later comparison
embeddings_before = model.embedding.weight.data[viz_indices].clone().cpu().numpy()
tsne_before = TSNE(n_components=2, random_state=42, perplexity=5)
reduced_before = tsne_before.fit_transform(embeddings_before)

fig, ax = plt.subplots(1, 1, figsize=(9, 7))
ax.scatter(reduced_before[:, 0], reduced_before[:, 1],
           c=viz_colors, s=180, alpha=0.7, edgecolors='white', linewidth=1.5)
for i, w in enumerate(viz_words):
    ax.annotate(w, (reduced_before[i, 0], reduced_before[i, 1]),
                fontsize=10, ha='center', va='bottom', xytext=(0, 6),
                textcoords='offset points')
ax.set_title("Before training: Randomly initialized Embedding (t-SNE reduction)", fontsize=13)
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()
print("-> Before training, words from the same semantic group are randomly scattered with no clustering pattern.")
print("-> The next cell starts training, so we can observe how word vectors change.")


In [ ]:
# Training loop + post-training t-SNE + before/after comparison
# nanoGPT style: batch -> forward -> loss -> backward -> clip grad -> step

import torch.nn.functional as F

batch_size = 64
n_steps = 300
losses = []

model.train()
for step in range(n_steps):
    # Randomly sample a batch
    idx = torch.randint(0, n_sequences, (batch_size,))
    x_batch = x_data[idx]
    y_batch = y_data[idx]

    # Forward
    logits = model(x_batch)                      # [batch, seq_len, vocab_size]
    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),          # [batch*seq_len, vocab_size]
        y_batch.reshape(-1)                      # [batch*seq_len]
    )

    # Backward + gradient clipping (standard nanoGPT practice)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    losses.append(loss.item())
    if (step + 1) % 60 == 0:
        print(f"  step {step+1:4d}/{n_steps}: loss = {loss.item():.4f}")

print(f"\nInitial loss: {losses[0]:.4f} -> Final loss: {losses[-1]:.4f}")
print(f"Loss reduction: {(losses[0] - losses[-1]) / losses[0] * 100:.1f}%")

# Step 1: Get post-training Embedding
model.eval()
with torch.no_grad():
    embeddings_after = model.embedding.weight.data[viz_indices].cpu().numpy()

# Step 2: Joint t-SNE -- concatenate pre- and post-training vectors for joint dimensionality reduction
all_embeddings = np.vstack([embeddings_before, embeddings_after])
tsne_joint = TSNE(n_components=2, random_state=42, perplexity=5)
all_reduced = tsne_joint.fit_transform(all_embeddings)

n_words = len(viz_words)
reduced_before_joint = all_reduced[:n_words]
reduced_after_joint = all_reduced[n_words:]

# Step 3: Three-panel comparison -- before training / after training / loss curve
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# Left panel: before training
axes[0].scatter(reduced_before_joint[:, 0], reduced_before_joint[:, 1],
                c=viz_colors, s=180, alpha=0.7, edgecolors='white', linewidth=1.5)
for i, w in enumerate(viz_words):
    axes[0].annotate(w, (reduced_before_joint[i, 0], reduced_before_joint[i, 1]),
                     fontsize=9, ha='center', va='bottom', xytext=(0, 5),
                     textcoords='offset points')
axes[0].set_title("Before training: Random distribution", fontsize=13)
axes[0].set_xticks([])
axes[0].set_yticks([])

# Middle panel: after training
axes[1].scatter(reduced_after_joint[:, 0], reduced_after_joint[:, 1],
                c=viz_colors, s=180, alpha=0.7, edgecolors='white', linewidth=1.5)
for i, w in enumerate(viz_words):
    axes[1].annotate(w, (reduced_after_joint[i, 0], reduced_after_joint[i, 1]),
                     fontsize=9, ha='center', va='bottom', xytext=(0, 5),
                     textcoords='offset points')
axes[1].set_title("After training: Semantic clustering emerges", fontsize=13)
axes[1].set_xticks([])
axes[1].set_yticks([])

# Right panel: loss curve
axes[2].plot(range(n_steps), losses, color='#2c3e50', linewidth=1.2)
axes[2].set_title("Training Loss", fontsize=13)
axes[2].set_xlabel("step")
axes[2].set_ylabel("cross-entropy loss")
axes[2].grid(True, alpha=0.3)

fig.suptitle("Embedding before/after comparison: Semantic clustering from scratch", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("embedding_training_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print("-> Figure saved as embedding_training_tsne.png")

# Step 4: Quantitative verification -- within-group similarity vs. cross-group similarity
from sklearn.metrics.pairwise import cosine_similarity

emb_after = model.embedding.weight.data.cpu().numpy()

# Calculate average cosine similarity within each semantic group
within_sims = []
for name, idx_list in viz_groups_actual.items():
    group_emb = emb_after[idx_list]
    sim_matrix = cosine_similarity(group_emb)
    triu_idx = np.triu_indices_from(sim_matrix, k=1)
    within_sims.extend(sim_matrix[triu_idx])

# Cross-group: numbers vs people (largest semantic difference)
if "numbers" in viz_groups_actual and "people" in viz_groups_actual:
    num_emb = emb_after[viz_groups_actual["numbers"]]
    ppl_emb = emb_after[viz_groups_actual["people"]]
    cross_sims = cosine_similarity(num_emb, ppl_emb).flatten()
    avg_within = np.mean(within_sims)
    avg_cross = np.mean(cross_sims)
    print(f"\nQuantitative verification:")
    print(f"  Within-group avg cosine similarity:        {avg_within:.4f}")
    print(f"  Cross-group avg cosine similarity (numbers vs people): {avg_cross:.4f}")
    print(f"  Within/cross ratio:                        {avg_within / avg_cross:.2f}x")
    print(f"  -> Within-group word vectors are indeed closer than cross-group -- semantic structure emerges from training.")

print(f"\nKey observations:")
print(f"  1. Left -> Middle: same-colored words go from scattered to clustered; Embedding learned semantic grouping.")
print(f"  2. Loss dropped from {losses[0]:.2f} to {losses[-1]:.2f} -- this is a real training signal on actual text.")
print(f"  3. No manual labeling was involved; clustering arises purely from the statistical pattern of 'similar context -> similar vectors'.")


## Summary

What we learned in this section:

- Token IDs are just labels and cannot be used directly as numerical input for the model -- the magnitude of an ID has no relationship to semantics
- Dense vectors use multiple real-valued dimensions to jointly describe a token, with a fixed size (d_model); the distance between vectors directly reflects semantic closeness
- A word's meaning is determined by its context -- words with similar meanings appear in similar contexts. This is the theoretical foundation for why Embedding can learn meaningful vectors
- Co-occurrence matrices are a count-based distributed representation; modern LLMs use trainable low-dimensional dense vectors instead
- `nn.Embedding` is a learnable [vocab_size, d_model] matrix; looking up a row retrieves the corresponding token vector. In end-to-end training, it is updated together with other parameters, driven by the loss
- Industry practices: weight tying is an optional design; weight decay depends on parameter grouping code (bias/Norm are usually excluded); initialization commonly uses small standard deviations like N(0, 0.02); mixed precision behavior depends on AMP and the training framework

The same token gets the same vector regardless of where it appears -- the model also needs to know each token's position in the sentence. The next section introduces Positional Encoding to solve this problem.


## Exercises

> You can ask an AI to help explain concepts or break down the approach, but avoid asking it to complete the exercises outright.


**Exercise 1: Embedding Lookup**

Token IDs themselves have no semantics; Embedding converts IDs into vectors.

Hint: `embedding_table[token_ids]` can retrieve multiple rows at once.


In [ ]:
# Exercise 1: Embedding lookup fill-in
import torch

embedding_table = torch.tensor([
    [1.0, 0.0],  # token 0
    [0.0, 1.0],  # token 1
    [1.0, 1.0],  # token 2
])
token_ids = torch.tensor([2, 0, 1])

# TODO: Replace the triple-quoted placeholder below with your code
vectors = """Replace this with code to look up vectors from embedding_table using token_ids"""

assert not isinstance(vectors, str), "Please replace the triple-quoted placeholder first"
expected = torch.tensor([[1.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
assert torch.equal(vectors, expected), vectors
print("✅ Exercise 1 passed: You understand that Embedding's core operation is a lookup.")


In [ ]:
import torch
import torch.nn as nn

# Exercise 2: Train a mini Embedding
# Goal: Run a training loop by hand and observe how Embedding vectors change

torch.manual_seed(42)

vocab_size, embed_dim = 5, 2
embedding = nn.Embedding(vocab_size, embed_dim)
optimizer = torch.optim.SGD(embedding.parameters(), lr=0.5)

# Record vectors before training
before = embedding.weight.data.clone()

# Training objective: make the vectors for token 0 and token 1 closer
for step in range(20):
    vec0 = embedding(torch.tensor(0))
    vec1 = embedding(torch.tensor(1))

    # TODO: Construct a loss that minimizes the distance between vec0 and vec1
    loss = """Replace this with code to compute the distance between vec0 and vec1 as the loss"""

    assert not isinstance(loss, str), "Please replace the placeholder first"
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Calculate distance change before and after training
d_before = (before[0] - before[1]).pow(2).sum().item()
d_after = (embedding.weight.data[0] - embedding.weight.data[1]).pow(2).sum().item()
print(f"Distance before training: {d_before:.4f}")
print(f"Distance after training:  {d_after:.4f}")
assert d_after < d_before, "After training, tokens 0 and 1 should be closer"
print("✅ Exercise 2 passed: You trained an Embedding by hand and experienced how gradients update vectors.")


**Exercise 2: Train a Mini Embedding**

Run a training loop by hand and observe how Embedding vectors change.

Goal: Make the vectors for token 0 and token 1 closer together.

Hint: The distance between two vectors can be computed with `(vec0 - vec1).pow(2).sum()`. Use it as the loss to optimize, and the distance will keep shrinking.


**Exercise 3: Real Embedding Model + t-SNE Visualization**

Use a pre-trained Embedding model from HuggingFace, select a set of words with clear semantic groupings, obtain their vectors, and use t-SNE to reduce them to 2D for visualization. Observe whether words from the same semantic group cluster together and whether different groups are far apart.

Select four groups of words: animals (cat, dog, elephant, tiger), countries (china, japan, france, germany), food (pizza, sushi, bread, noodle), technology (computer, laptop, phone, tablet). 16 words total, each encoded into a dense vector by the model.

Install dependencies first:

```bash
pip install sentence-transformers scikit-learn
```

Hint: `TSNE(n_components=2, random_state=42, perplexity=5)` creates a t-SNE object, and `fit_transform(embeddings)` performs the dimensionality reduction.


In [ ]:
# Exercise 3: Real Embedding model + t-SNE visualization
# Goal: Use a HuggingFace pre-trained Embedding model to get word vectors,
# reduce to 2D with t-SNE and visualize, verifying that synonyms cluster
# and dissimilar words are far apart.

# If dependencies are not installed, uncomment the following line:
# !pip install sentence-transformers scikit-learn -q

import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Load HuggingFace pre-trained Embedding model
# all-MiniLM-L6-v2 is a lightweight 384-dimensional English model trained on 1B+ sentence pairs
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded, Embedding dimension: {model.get_sentence_embedding_dimension()}")

# Step 2: Select words -- four semantically different groups, semantically similar within each group
words = [
    # Animals
    "cat", "dog", "elephant", "tiger",
    # Countries
    "china", "japan", "france", "germany",
    # Food
    "pizza", "sushi", "bread", "noodle",
    # Technology
    "computer", "laptop", "phone", "tablet",
]
groups = {"Animals": (0, 4), "Countries": (4, 8), "Food": (8, 12), "Technology": (12, 16)}

# Step 3: Get Embedding vectors for each word
embeddings = model.encode(words)
print(f"Embedding shape: {embeddings.shape}  <- [{len(words)} words, each {embeddings.shape[1]} dimensions]")

# Step 4: t-SNE reduction to 2D
# TODO: Use TSNE to reduce high-dimensional Embedding to a 2D plane
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
reduced = """Replace this with tsne.fit_transform(embeddings) to reduce dimensions"""

assert not isinstance(reduced, str), "Please replace the triple-quoted placeholder first"
assert reduced.shape == (len(words), 2), \
    f"Reduced shape should be ({len(words)}, 2), got {reduced.shape}"

# Step 5: Visualize -- same color for same group, annotate word names
colors = {"Animals": "#E74C3C", "Countries": "#3498DB", "Food": "#2ECC71", "Technology": "#F39C12"}

plt.figure(figsize=(12, 9))
for group_name, (start, end) in groups.items():
    group_words = words[start:end]
    group_points = reduced[start:end]
    plt.scatter(group_points[:, 0], group_points[:, 1],
                c=colors[group_name], s=200, alpha=0.7, label=group_name,
                edgecolors="white", linewidth=1.5)
    for word, (x, y) in zip(group_words, group_points):
        plt.annotate(word, (x, y), fontsize=11, ha="center", va="bottom",
                     xytext=(0, 8), textcoords="offset points")

plt.title("t-SNE visualization: Semantic clustering of pre-trained Embedding word vectors", fontsize=14)
plt.legend(fontsize=11, loc="lower right")
plt.tight_layout()
plt.savefig("embedding_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print("-> Figure saved as embedding_tsne.png")

# Step 6: Quantitative verification -- within-group similarity vs. cross-group similarity
within_sim = []
for start, end in groups.values():
    group_emb = embeddings[start:end]
    sim_matrix = cosine_similarity(group_emb)
    # Take upper triangular elements (exclude diagonal self-similarity of 1.0)
    triu_idx = np.triu_indices_from(sim_matrix, k=1)
    within_sim.extend(sim_matrix[triu_idx])

# Cross-group: Animals(0:4) vs Technology(12:16)
cross_sim = cosine_similarity(embeddings[0:4], embeddings[12:16]).flatten()

avg_within = np.mean(within_sim)
avg_cross = np.mean(cross_sim)

print(f"\nKey observations:")
print(f"  Within-group avg cosine similarity:             {avg_within:.4f}")
print(f"  Cross-group avg cosine similarity (Animals vs Technology): {avg_cross:.4f}")
print(f"  Within/cross ratio:                             {avg_within / avg_cross:.2f}x")
print(f"  -> Within-group word vectors are indeed closer -- Embedding captures semantic similarity.")

assert avg_within > avg_cross, \
    f"Within-group similarity ({avg_within:.4f}) should be higher than cross-group ({avg_cross:.4f})"
print(f"✅ Exercise 3 passed: You verified that semantically similar = vector similarity using a real pre-trained Embedding model.")


## References

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), 2017 -- Original Transformer paper; the convention of multiplying Embedding by sqrt(d_model) comes from this paper
- Harvard NLP, [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) -- Line-by-line implementation of the original paper
- Mikolov et al., [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781), 2013 -- Word2Vec, classic work on distributed representations
- Karpathy, [nanoGPT](https://github.com/karpathy/nanoGPT/blob/master/model.py) -- The `configure_optimizers` method shows Embedding weights participating in weight decay as usual, providing direct code evidence for end-to-end training
- Chen et al., [Stable Language Model Pre-training by Reducing Embedding Variability](https://aclanthology.org/2024.emnlp-main.606.pdf), EMNLP 2024 -- Mathematically derives the gradient formula for the token embedding layer, proving its gradient norm is largest under Pre-LN architecture
- Takase et al., [Spike No More: Stabilizing the Pre-training of Large Language Models](https://openreview.net/forum?id=52YBEzcI0l), COLM 2025 -- Proposes adding LayerNorm after Embedding to prevent gradient explosion
- Wang et al., [Text Embeddings by Weakly-Supervised Contrastive Pre-training](https://arxiv.org/abs/2212.03533), 2022 -- E5: Weakly-supervised contrastive pre-training paradigm
- Xiao et al., [C-Pack: Packaged Resources To Advance General Chinese Embedding](https://arxiv.org/abs/2309.07597), 2023 -- BGE: RetroMAE pre-training + contrastive fine-tuning recipe
- Gunther et al., [jina-embeddings-v3: Multilingual Embeddings With Task LoRA](https://arxiv.org/abs/2409.10173), 2024 -- Jina Embeddings v3: Multi-task contrastive learning + LoRA adapter
- Zhang et al., [Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models](https://arxiv.org/abs/2506.05176), 2025 -- Three-stage Embedding training based on Qwen3 foundation LLM
- Chen et al., [Scaling Embedding Layers in Language Models](https://arxiv.org/abs/2502.01637), NeurIPS 2025 -- Training efficiency research on large-scale Embedding layers
